In [6]:
import numpy as np
import pandas as pd
import soundfile as sf
import librosa as li
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    'dataset_root': 'dataset/physical_model/new/',
    'preprocessed_root': 'preprocessed/physical_model/new/',
    'variations': [
        'convolved_Bernardel_ear_norm',
        'convolved_Bernardel_ff_norm',
        'convolved_StradCopy_ear_norm',
        'convolved_StradCopy_ff_norm',
        'convolved_Sundin_ear_norm',
        'convolved_Sundin_ff_norm',
    ],
    'strings': ['A', 'D', 'E', 'G'],
    'splits': ['full', 'A', 'D', 'E', 'G'],
    'sampling_rate': 16000,
    'block_size': 64,
    'signal_length': 64000,
    'n_validation_samples': 20,
    'rtol': 1e-5,
    'atol': 1e-7,
    'manifest_output_dir': 'preprocessing_manifests',
}

def check_audio_lengths(variation, config=CONFIG):
    dataset_path = Path(config['dataset_root']) / variation
    
    print(f"\nChecking audio lengths for: {variation}")
    print("="*80)
    
    all_files = []
    for string in config['strings']:
        string_path = dataset_path / string
        if string_path.exists():
            files = sorted(list(string_path.glob("*.wav")))
            all_files.extend(files)
    
    print(f"Total files found: {len(all_files)}")
    
    if len(all_files) == 0:
        print("No files found!")
        return None
    
    lengths = []
    sample_rates = []
    
    for file_path in tqdm(all_files[:100], desc="Sampling files"):
        audio, sr = sf.read(file_path)
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)
        lengths.append(len(audio))
        sample_rates.append(sr)
    
    lengths = np.array(lengths)
    sample_rates = np.array(sample_rates)
    
    print(f"\nSample rates: unique={np.unique(sample_rates)}")
    print(f"Audio lengths (samples): min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.1f}")
    print(f"Audio lengths (seconds @ original SR): min={lengths.min()/sample_rates[0]:.3f}, max={lengths.max()/sample_rates[0]:.3f}")
    
    resampled_lengths = (lengths * config['sampling_rate'] / sample_rates).astype(int)
    print(f"Resampled lengths (@ 16kHz): min={resampled_lengths.min()}, max={resampled_lengths.max()}, mean={resampled_lengths.mean():.1f}")
    print(f"Signal length config: {config['signal_length']}")
    print(f"Expected segments per file: {np.ceil(resampled_lengths / config['signal_length']).astype(int)}")
    
    all_same = len(np.unique(lengths)) == 1
    print(f"\nAll files same length: {all_same}")
    
    return {
        'n_files': len(all_files),
        'lengths': lengths,
        'sample_rates': sample_rates,
        'all_same_length': all_same
    }

def create_preprocessing_manifest(variation, split, config=CONFIG):
    dataset_path = Path(config['dataset_root']) / variation
    
    print(f"\nCreating preprocessing manifest for: {variation} / {split}")
    print("="*80)
    
    if split == 'full':
        all_files = []
        for string in config['strings']:
            string_path = dataset_path / string
            if string_path.exists():
                files = sorted(list(string_path.glob("*.wav")))
                all_files.extend([(f, string) for f in files])
        all_files.sort(key=lambda x: x[0])
    else:
        string_path = dataset_path / split
        if not string_path.exists():
            print(f"String directory not found: {string_path}")
            return None
        files = sorted(list(string_path.glob("*.wav")))
        all_files = [(f, split) for f in files]
    
    print(f"Total files: {len(all_files)}")
    
    if len(all_files) == 0:
        print("No files found!")
        return None
    
    manifest_data = []
    cumulative_segments = 0
    
    for idx, (file_path, string) in enumerate(tqdm(all_files, desc="Processing files")):
        try:
            audio, sr = sf.read(file_path)
            if len(audio.shape) > 1:
                audio = audio.mean(axis=1)
            
            if sr != config['sampling_rate']:
                audio = li.resample(audio, orig_sr=sr, target_sr=config['sampling_rate'])
            
            pad = (config['signal_length'] - len(audio) % config['signal_length']) % config['signal_length']
            if pad:
                audio = np.pad(audio, (0, pad))
            
            n_segments = audio.shape[0] // config['signal_length']
            
            manifest_data.append({
                'file_index': idx,
                'filename': file_path.name,
                'string': string,
                'relative_path': str(file_path.relative_to(dataset_path)),
                'n_segments': n_segments,
                'start_segment_idx': cumulative_segments,
                'end_segment_idx': cumulative_segments + n_segments - 1,
                'original_length': len(audio) - pad,
                'padded_length': len(audio),
            })
            
            cumulative_segments += n_segments
            
        except Exception as e:
            print(f"\nError processing {file_path}: {e}")
            continue
    
    df_manifest = pd.DataFrame(manifest_data)
    
    print(f"\nManifest summary:")
    print(f"  Total files: {len(df_manifest)}")
    print(f"  Total segments: {cumulative_segments}")
    print(f"  Segments per file: min={df_manifest['n_segments'].min()}, max={df_manifest['n_segments'].max()}, mean={df_manifest['n_segments'].mean():.2f}")
    print(f"  Files per string: {df_manifest['string'].value_counts().to_dict()}")
    
    return df_manifest

def validate_preprocessing(variation, split, manifest, config=CONFIG):
    dataset_path = Path(config['dataset_root']) / variation
    preprocessed_path = Path(config['preprocessed_root']) / variation / split
    
    signals_path = preprocessed_path / "signals.npy"
    pitches_path = preprocessed_path / "pitches.npy"
    loudness_path = preprocessed_path / "loudness.npy"
    
    print(f"\nValidating preprocessing for: {variation} / {split}")
    print("="*80)
    
    if not signals_path.exists():
        print(f"Signals file not found: {signals_path}")
        return None
    
    signals = np.load(signals_path)
    pitches = np.load(pitches_path)
    loudness = np.load(loudness_path)
    
    print(f"Loaded arrays:")
    print(f"  signals.npy: shape={signals.shape}, dtype={signals.dtype}")
    print(f"  pitches.npy: shape={pitches.shape}, dtype={pitches.dtype}")
    print(f"  loudness.npy: shape={loudness.shape}, dtype={loudness.dtype}")
    
    expected_segments = manifest['start_segment_idx'].max() + manifest['n_segments'].iloc[-1]
    print(f"\nExpected total segments from manifest: {expected_segments}")
    print(f"Actual segments in .npy: {signals.shape[0]}")
    print(f"Match: {expected_segments == signals.shape[0]}")
    
    n_validation = min(config['n_validation_samples'], len(manifest))
    validation_indices = np.random.choice(len(manifest), n_validation, replace=False)
    
    results = []
    
    print(f"\nValidating {n_validation} random samples...")
    print("="*80)
    
    for val_idx in tqdm(validation_indices):
        file_info = manifest.iloc[val_idx]
        file_path = dataset_path / file_info['relative_path']
        
        try:
            audio, sr = sf.read(file_path)
            if len(audio.shape) > 1:
                audio = audio.mean(axis=1)
            
            if sr != config['sampling_rate']:
                audio = li.resample(audio, orig_sr=sr, target_sr=config['sampling_rate'])
            
            pad = (config['signal_length'] - len(audio) % config['signal_length']) % config['signal_length']
            if pad:
                audio = np.pad(audio, (0, pad))
            
            audio_segments = audio.reshape(-1, config['signal_length']).astype(np.float32)
            
            start_idx = file_info['start_segment_idx']
            end_idx = file_info['end_segment_idx']
            
            stored_segments = signals[start_idx:end_idx+1]
            
            signals_match = np.allclose(audio_segments, stored_segments, rtol=config['rtol'], atol=config['atol'])
            max_diff = np.abs(audio_segments - stored_segments).max()
            
            results.append({
                'file_index': file_info['file_index'],
                'filename': file_info['filename'],
                'string': file_info['string'],
                'segment_indices': f"{start_idx}-{end_idx}",
                'signals_match': signals_match,
                'max_signal_diff': max_diff,
            })
            
        except Exception as e:
            print(f"\nError validating {file_info['filename']}: {e}")
            results.append({
                'file_index': file_info['file_index'],
                'filename': file_info['filename'],
                'string': file_info['string'],
                'segment_indices': f"{start_idx}-{end_idx}",
                'signals_match': False,
                'max_signal_diff': np.nan,
                'error': str(e)
            })
    
    df_validation = pd.DataFrame(results)
    
    print("\nValidation Results:")
    print("="*80)
    print(f"Samples validated: {len(df_validation)}")
    print(f"Signals matched: {df_validation['signals_match'].sum()}/{len(df_validation)}")
    print(f"Max signal difference across all samples: {df_validation['max_signal_diff'].max():.2e}")
    
    if not df_validation['signals_match'].all():
        print("\nFailed validations:")
        print(df_validation[~df_validation['signals_match']][['filename', 'max_signal_diff']])
    else:
        print("\n✓ All validations passed!")
    
    return df_validation

print("\n" + "#"*80)
print("# STEP 1: PREPROCESSING VERIFICATION")
print("#"*80)

for variation in CONFIG['variations']:
    check_audio_lengths(variation, CONFIG)

manifests = {}
validation_results = {}

for variation in CONFIG['variations']:
    for split in CONFIG['splits']:
        key = f"{variation}_{split}"
        manifests[key] = create_preprocessing_manifest(variation, split, CONFIG)
        
        if manifests[key] is not None:
            validation_results[key] = validate_preprocessing(variation, split, manifests[key], CONFIG)

output_path = Path(CONFIG['manifest_output_dir'])
output_path.mkdir(exist_ok=True)

print(f"\nSaving manifests to: {CONFIG['manifest_output_dir']}")
print("="*80)

for key, manifest in manifests.items():
    if manifest is not None:
        output_file = output_path / f"{key}_manifest.csv"
        manifest.to_csv(output_file, index=False)
        print(f"✓ Saved: {output_file.name} ({len(manifest)} files)")

print("\n" + "="*80)
print("STEP 1 COMPLETE: PREPROCESSING VERIFICATION")
print("="*80)
print("\nSummary:")
for key in manifests.keys():
    if key in validation_results and validation_results[key] is not None:
        n_validated = len(validation_results[key])
        n_passed = validation_results[key]['signals_match'].sum()
        status = "✓ PASS" if n_passed == n_validated else "✗ FAIL"
        print(f"  {status} {key}: {n_passed}/{n_validated} samples matched")
print("\nManifest files saved in: preprocessing_manifests/")
print("Ready to proceed to Step 2: Parameter Extraction")
print("="*80)


################################################################################
# STEP 1: PREPROCESSING VERIFICATION
################################################################################

Checking audio lengths for: convolved_Bernardel_ear_norm
Total files found: 4000


Sampling files: 100%|██████████| 100/100 [00:00<00:00, 185.92it/s]



Sample rates: unique=[44100]
Audio lengths (samples): min=88200, max=88200, mean=88200.0
Audio lengths (seconds @ original SR): min=2.000, max=2.000
Resampled lengths (@ 16kHz): min=32000, max=32000, mean=32000.0
Signal length config: 64000
Expected segments per file: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]

All files same length: True

Checking audio lengths for: convolved_Bernardel_ff_norm
Total files found: 4000


Sampling files: 100%|██████████| 100/100 [00:00<00:00, 178.29it/s]



Sample rates: unique=[44100]
Audio lengths (samples): min=88200, max=88200, mean=88200.0
Audio lengths (seconds @ original SR): min=2.000, max=2.000
Resampled lengths (@ 16kHz): min=32000, max=32000, mean=32000.0
Signal length config: 64000
Expected segments per file: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]

All files same length: True

Checking audio lengths for: convolved_StradCopy_ear_norm
Total files found: 4000


...


Validation Results:
Samples validated: 20
Signals matched: 20/20
Max signal difference across all samples: 0.00e+00

✓ All validations passed!

Creating preprocessing manifest for: convolved_Sundin_ff_norm / G
Total files: 1000


Processing files: 100%|██████████| 1000/1000 [00:00<00:00, 1091.47it/s]



Manifest summary:
  Total files: 1000
  Total segments: 1000
  Segments per file: min=1, max=1, mean=1.00
  Files per string: {'G': 1000}

Validating preprocessing for: convolved_Sundin_ff_norm / G
Loaded arrays:
  signals.npy: shape=(1000, 64000), dtype=float32
  pitches.npy: shape=(1000, 1000), dtype=float32
  loudness.npy: shape=(1000, 1000), dtype=float32

Expected total segments from manifest: 1000
Actual segments in .npy: 1000
Match: True

Validating 20 random samples...


100%|██████████| 20/20 [00:00<00:00, 1112.09it/s]



Validation Results:
Samples validated: 20
Signals matched: 20/20
Max signal difference across all samples: 0.00e+00

✓ All validations passed!

Saving manifests to: preprocessing_manifests
✓ Saved: convolved_Bernardel_ear_norm_full_manifest.csv (4000 files)
✓ Saved: convolved_Bernardel_ear_norm_A_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ear_norm_D_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ear_norm_E_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ear_norm_G_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ff_norm_full_manifest.csv (4000 files)
✓ Saved: convolved_Bernardel_ff_norm_A_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ff_norm_D_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ff_norm_E_manifest.csv (1000 files)
✓ Saved: convolved_Bernardel_ff_norm_G_manifest.csv (1000 files)
✓ Saved: convolved_StradCopy_ear_norm_full_manifest.csv (4000 files)
✓ Saved: convolved_StradCopy_ear_norm_A_manifest.csv (1000 files)
✓ Saved: convo

In [7]:
import torch
import numpy as np
import pandas as pd
import yaml
import re
from pathlib import Path
from tqdm import tqdm
import warnings
from contextlib import redirect_stdout
import io
from ddsp_torch.model import DDSP

warnings.filterwarnings('ignore')

CONFIG_STEP2 = {
    'runs_root': 'runs/',
    'preprocessed_root': 'preprocessed/physical_model/new/',
    'manifest_dir': 'preprocessing_manifests',
    'output_dir': 'model_parameters',
    'variations': [
        'convolved_Bernardel_ear_norm',
        'convolved_Bernardel_ff_norm',
        'convolved_StradCopy_ear_norm',
        'convolved_StradCopy_ff_norm',
        'convolved_Sundin_ear_norm',
        'convolved_Sundin_ff_norm',
    ],
    'splits': ['full', 'A', 'D', 'E', 'G'],
    'physics_models': [
        {'suffix': '__vio', 'label': 'Physics (Baseline)'},
        {'suffix': '__vio_vpl_w1', 'label': 'Physics + VPL(w=1)'},
        {'suffix': '__vio_vpl_w100', 'label': 'Physics + VPL(w=100)'},
    ],
    'sampling_rate': 16000,
    'block_size': 64,
    'onset_exclusion_frames': 37,
}

def parse_filename(filename):
    pattern = r'([ADEG])_(\d+\.\d+)p_(\d+\.\d+)N_(\d+\.\d+)v_(\d+\.\d+)x'
    match = re.match(pattern, filename)
    if not match:
        return None
    return {
        'string': match.group(1),
        'bow_pos_pct': float(match.group(2)),
        'force_N': float(match.group(3)),
        'speed_ms': float(match.group(4)),
        'x2': float(match.group(5)),
    }

def get_run_name(variation, split, model_suffix):
    variation_short = variation.replace('convolved_', '')
    return f"{variation_short}_{split}{model_suffix}"

def load_model_silently(run_name, config=CONFIG_STEP2):
    run_dir = Path(config['runs_root']) / run_name
    model_path = run_dir / "final_state.pth"
    config_path = run_dir / "config.yaml"
    
    if not model_path.exists() or not config_path.exists():
        return None, None
    
    try:
        with open(config_path, 'r') as f:
            model_config = yaml.safe_load(f)
        
        with redirect_stdout(io.StringIO()):
            model = DDSP(**model_config.get("model", {}))
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.eval()
        
        return model, model_config
    except Exception as e:
        print(f"    Error loading model: {e}")
        return None, None

def extract_parameters_for_model(variation, split, model_suffix, config=CONFIG_STEP2):
    print(f"\n{'='*80}")
    print(f"Processing: {variation} / {split} / {model_suffix}")
    print(f"{'='*80}")
    
    run_name = get_run_name(variation, split, model_suffix)
    print(f"  Run name: {run_name}")
    
    model, model_config = load_model_silently(run_name, config)
    if model is None:
        print(f"  ✗ Model not found or failed to load")
        return None
    print(f"  ✓ Model loaded")
    
    manifest_path = Path(config['manifest_dir']) / f"{variation}_{split}_manifest.csv"
    if not manifest_path.exists():
        print(f"  ✗ Manifest not found: {manifest_path}")
        return None
    
    manifest = pd.read_csv(manifest_path)
    print(f"  ✓ Manifest loaded: {len(manifest)} files")
    
    if len(manifest) == 0:
        print(f"  ✗ No files in manifest")
        return None
    
    preprocessed_path = Path(config['preprocessed_root']) / variation / split
    signals_path = preprocessed_path / "signals.npy"
    pitches_path = preprocessed_path / "pitches.npy"
    loudness_path = preprocessed_path / "loudness.npy"
    
    if not signals_path.exists():
        print(f"  ✗ Preprocessed data not found: {signals_path}")
        return None
    
    signals = np.load(signals_path)
    pitches = np.load(pitches_path)
    loudness = np.load(loudness_path)
    
    print(f"  ✓ Loaded preprocessed arrays: {signals.shape[0]} segments")
    
    n_files = len(manifest)
    n_frames = pitches.shape[1]
    
    beta_all = np.zeros((n_files, n_frames), dtype=np.float32)
    gamma_all = np.zeros((n_files, n_frames), dtype=np.float32)
    alpha_all = np.zeros((n_files, n_frames), dtype=np.float32)
    B_all = np.zeros((n_files, n_frames), dtype=np.float32)
    
    ground_truth = {
        'filename': [],
        'string': [],
        'bow_pos_pct': [],
        'force_N': [],
        'speed_ms': [],
        'x2': [],
        'file_index': [],
        'segment_index': [],
    }
    
    print(f"  Processing {n_files} files...")
    
    successful = 0
    failed = 0
    
    for idx in tqdm(range(n_files), desc="  Inference"):
        file_info = manifest.iloc[idx]
        
        gt_params = parse_filename(file_info['filename'])
        if gt_params is None:
            failed += 1
            continue
        
        segment_idx = file_info['start_segment_idx']
        
        try:
            pitch_tensor = torch.from_numpy(pitches[segment_idx]).unsqueeze(0).unsqueeze(-1).float()
            loudness_tensor = torch.from_numpy(loudness[segment_idx]).unsqueeze(0).unsqueeze(-1).float()
            audio_tensor = torch.from_numpy(signals[segment_idx]).unsqueeze(0).float()
            
            with torch.no_grad():
                outputs = model(pitch_tensor, loudness_tensor, audio=audio_tensor)
            
            if 'violin_diagnostics' not in outputs or outputs['violin_diagnostics'] is None:
                failed += 1
                continue
            
            diagnostics = outputs['violin_diagnostics']
            
            beta_frames = diagnostics['β'].squeeze().cpu().numpy()
            gamma_frames = diagnostics['γ'].squeeze().cpu().numpy()
            alpha_frames = diagnostics['α'].squeeze().cpu().numpy()
            B_frames = diagnostics['B'].squeeze().cpu().numpy()
            
            beta_all[successful] = beta_frames
            gamma_all[successful] = gamma_frames
            alpha_all[successful] = alpha_frames
            B_all[successful] = B_frames
            
            ground_truth['filename'].append(file_info['filename'])
            ground_truth['string'].append(gt_params['string'])
            ground_truth['bow_pos_pct'].append(gt_params['bow_pos_pct'])
            ground_truth['force_N'].append(gt_params['force_N'])
            ground_truth['speed_ms'].append(gt_params['speed_ms'])
            ground_truth['x2'].append(gt_params['x2'])
            ground_truth['file_index'].append(file_info['file_index'])
            ground_truth['segment_index'].append(segment_idx)
            
            successful += 1
            
        except Exception as e:
            failed += 1
            continue
    
    if successful == 0:
        print(f"  ✗ No successful extractions")
        return None
    
    beta_all = beta_all[:successful]
    gamma_all = gamma_all[:successful]
    alpha_all = alpha_all[:successful]
    B_all = B_all[:successful]
    
    print(f"  ✓ Successfully processed: {successful}/{n_files} files")
    if failed > 0:
        print(f"  ⚠ Failed: {failed} files")
    
    output_data = {
        'beta': beta_all,
        'gamma': gamma_all,
        'alpha': alpha_all,
        'B': B_all,
        'filename': np.array(ground_truth['filename'], dtype=object),
        'string': np.array(ground_truth['string'], dtype=object),
        'bow_pos_pct': np.array(ground_truth['bow_pos_pct'], dtype=np.float32),
        'force_N': np.array(ground_truth['force_N'], dtype=np.float32),
        'speed_ms': np.array(ground_truth['speed_ms'], dtype=np.float32),
        'x2': np.array(ground_truth['x2'], dtype=np.float32),
        'file_index': np.array(ground_truth['file_index'], dtype=np.int32),
        'segment_index': np.array(ground_truth['segment_index'], dtype=np.int32),
        'n_frames': np.array(n_frames, dtype=np.int32),
        'onset_exclusion_frames': np.array(config['onset_exclusion_frames'], dtype=np.int32),
    }
    
    variation_short = variation.replace('convolved_', '')
    output_filename = f"{variation_short}_{split}{model_suffix}.npz"
    output_path = Path(config['output_dir']) / output_filename
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_path, **output_data)
    
    print(f"  ✓ Saved: {output_filename}")
    
    return output_path

print("\n" + "#"*80)
print("# STEP 2: PARAMETER EXTRACTION")
print("#"*80)

output_dir = Path(CONFIG_STEP2['output_dir'])
output_dir.mkdir(exist_ok=True)

extraction_results = []

total_configs = len(CONFIG_STEP2['variations']) * len(CONFIG_STEP2['splits']) * len(CONFIG_STEP2['physics_models'])
current = 0

for variation in CONFIG_STEP2['variations']:
    print(f"\n{'#'*80}")
    print(f"# VARIATION: {variation}")
    print(f"{'#'*80}")
    
    for split in CONFIG_STEP2['splits']:
        for model_spec in CONFIG_STEP2['physics_models']:
            current += 1
            print(f"\n[{current}/{total_configs}]")
            
            result_path = extract_parameters_for_model(
                variation, 
                split, 
                model_spec['suffix'], 
                CONFIG_STEP2
            )
            
            if result_path is not None:
                extraction_results.append({
                    'variation': variation,
                    'split': split,
                    'model_suffix': model_spec['suffix'],
                    'model_label': model_spec['label'],
                    'output_path': str(result_path),
                    'status': 'success'
                })
            else:
                extraction_results.append({
                    'variation': variation,
                    'split': split,
                    'model_suffix': model_spec['suffix'],
                    'model_label': model_spec['label'],
                    'output_path': None,
                    'status': 'failed'
                })

df_results = pd.DataFrame(extraction_results)

results_summary_path = output_dir / 'extraction_summary.csv'
df_results.to_csv(results_summary_path, index=False)

print("\n" + "="*80)
print("STEP 2 COMPLETE: PARAMETER EXTRACTION")
print("="*80)
print(f"\nTotal configurations processed: {len(df_results)}")
print(f"Successful: {(df_results['status'] == 'success').sum()}")
print(f"Failed: {(df_results['status'] == 'failed').sum()}")
print(f"\nOutput directory: {output_dir}/")
print(f"Summary saved: {results_summary_path.name}")
print("\nFiles saved as NPZ format with:")
print("  - Frame-wise parameters: beta, gamma, alpha, B")
print("  - Ground truth: bow_pos_pct, force_N, speed_ms, x2, string")
print("  - Metadata: filename, file_index, segment_index, n_frames, onset_exclusion_frames")
print("\nReady to proceed to Step 3: Correlation Analysis")
print("="*80)


################################################################################
# STEP 2: PARAMETER EXTRACTION
################################################################################

################################################################################
# VARIATION: convolved_Bernardel_ear_norm
################################################################################

[1/90]

Processing: convolved_Bernardel_ear_norm / full / __vio
  Run name: Bernardel_ear_norm_full__vio
  ✓ Model loaded
  ✓ Manifest loaded: 4000 files
  ✓ Loaded preprocessed arrays: 4000 segments
  Processing 4000 files...


  Inference: 100%|██████████| 4000/4000 [04:43<00:00, 14.10it/s]


  ✓ Successfully processed: 4000/4000 files
  ✓ Saved: Bernardel_ear_norm_full__vio.npz

[2/90]

Processing: convolved_Bernardel_ear_norm / full / __vio_vpl_w1
  Run name: Bernardel_ear_norm_full__vio_vpl_w1
  ✓ Model loaded
  ✓ Manifest loaded: 4000 files
  ✓ Loaded preprocessed arrays: 4000 segments
  Processing 4000 files...


  Inference: 100%|██████████| 4000/4000 [04:39<00:00, 14.29it/s]


  ✓ Successfully processed: 4000/4000 files
  ✓ Saved: Bernardel_ear_norm_full__vio_vpl_w1.npz

[3/90]

Processing: convolved_Bernardel_ear_norm / full / __vio_vpl_w100
  Run name: Bernardel_ear_norm_full__vio_vpl_w100
  ✓ Model loaded
  ✓ Manifest loaded: 4000 files
  ✓ Loaded preprocessed arrays: 4000 segments
  Processing 4000 files...


...

  ✓ Successfully processed: 1000/1000 files
  ✓ Saved: Sundin_ff_norm_G__vio.npz

[89/90]

Processing: convolved_Sundin_ff_norm / G / __vio_vpl_w1
  Run name: Sundin_ff_norm_G__vio_vpl_w1
  ✓ Model loaded
  ✓ Manifest loaded: 1000 files
  ✓ Loaded preprocessed arrays: 1000 segments
  Processing 1000 files...


  Inference: 100%|██████████| 1000/1000 [01:09<00:00, 14.30it/s]


  ✓ Successfully processed: 1000/1000 files
  ✓ Saved: Sundin_ff_norm_G__vio_vpl_w1.npz

[90/90]

Processing: convolved_Sundin_ff_norm / G / __vio_vpl_w100
  Run name: Sundin_ff_norm_G__vio_vpl_w100
  ✓ Model loaded
  ✓ Manifest loaded: 1000 files
  ✓ Loaded preprocessed arrays: 1000 segments
  Processing 1000 files...


  Inference: 100%|██████████| 1000/1000 [01:09<00:00, 14.38it/s]


  ✓ Successfully processed: 1000/1000 files
  ✓ Saved: Sundin_ff_norm_G__vio_vpl_w100.npz

STEP 2 COMPLETE: PARAMETER EXTRACTION

Total configurations processed: 90
Successful: 90
Failed: 0

Output directory: model_parameters/
Summary saved: extraction_summary.csv

Files saved as NPZ format with:
  - Frame-wise parameters: beta, gamma, alpha, B
  - Ground truth: bow_pos_pct, force_N, speed_ms, x2, string
  - Metadata: filename, file_index, segment_index, n_frames, onset_exclusion_frames

Ready to proceed to Step 3: Correlation Analysis


In [13]:
import torch
import numpy as np
import yaml
from pathlib import Path
from contextlib import redirect_stdout
import io
from ddsp_torch.model import DDSP

def inspect_model_outputs():
    # Configuration
    runs_root = 'runs/'
    preprocessed_root = 'preprocessed/physical_model/new/'
    variation = 'Bernardel_ear_norm'
    split = 'full'
    model_suffix = '__vio'
    
    # Construct paths
    variation_short = variation.replace('convolved_', '')
    run_name = f"{variation_short}_{split}{model_suffix}"
    run_dir = Path(runs_root) / run_name
    
    print(f"Loading model: {run_name}")
    print("="*80)
    
    # Load model
    model_path = run_dir / "final_state.pth"
    config_path = run_dir / "config.yaml"
    
    with open(config_path, 'r') as f:
        model_config = yaml.safe_load(f)
    
    with redirect_stdout(io.StringIO()):
        model = DDSP(**model_config.get("model", {}))
        model.load_state_dict(torch.load(model_path, map_location='cpu'))
        model.eval()
    
    print("✓ Model loaded\n")
    
    # Load one sample
    preprocessed_path = Path(preprocessed_root) / f"convolved_{variation}" / split
    signals = np.load(preprocessed_path / "signals.npy")
    pitches = np.load(preprocessed_path / "pitches.npy")
    loudness = np.load(preprocessed_path / "loudness.npy")
    
    print(f"Preprocessed data shapes:")
    print(f"  signals: {signals.shape}")
    print(f"  pitches: {pitches.shape}")
    print(f"  loudness: {loudness.shape}")
    
    # Prepare tensors for first sample
    pitch_tensor = torch.from_numpy(pitches[0]).unsqueeze(0).unsqueeze(-1).float()
    loudness_tensor = torch.from_numpy(loudness[0]).unsqueeze(0).unsqueeze(-1).float()
    audio_tensor = torch.from_numpy(signals[0]).unsqueeze(0).float()
    
    print(f"\nInput tensor shapes:")
    print(f"  pitch: {pitch_tensor.shape}")
    print(f"  loudness: {loudness_tensor.shape}")
    print(f"  audio: {audio_tensor.shape}")
    
    # Run inference
    print(f"\n{'='*80}")
    print("Running inference...")
    print("="*80)
    
    with torch.no_grad():
        outputs = model(pitch_tensor, loudness_tensor, audio=audio_tensor)
    
    # Inspect outputs
    print(f"\n{'='*80}")
    print("OUTPUTS INSPECTION")
    print("="*80)
    
    print(f"\nTop-level keys in outputs:")
    for key in outputs.keys():
        value = outputs[key]
        if torch.is_tensor(value):
            print(f"  {key}: tensor, shape={value.shape}")
        elif value is None:
            print(f"  {key}: None")
        elif isinstance(value, dict):
            print(f"  {key}: dict with {len(value)} keys")
        else:
            print(f"  {key}: {type(value)}")
    
    # Deep dive into violin_diagnostics
    if 'violin_diagnostics' in outputs and outputs['violin_diagnostics'] is not None:
        print(f"\n{'='*80}")
        print("VIOLIN_DIAGNOSTICS INSPECTION")
        print("="*80)
        
        diagnostics = outputs['violin_diagnostics']
        
        print(f"\nKeys in violin_diagnostics:")
        for key in diagnostics.keys():
            print(f"  - {key}")
        
        print(f"\n{'='*80}")
        print("PARAMETER DETAILS")
        print("="*80)
        
        for key, value in diagnostics.items():
            if torch.is_tensor(value):
                print(f"\n{key}:")
                print(f"  Shape: {value.shape}")
                print(f"  Dtype: {value.dtype}")
                print(f"  Min: {value.min().item():.6f}")
                print(f"  Max: {value.max().item():.6f}")
                print(f"  Mean: {value.mean().item():.6f}")
                print(f"  Std: {value.std().item():.6f}")
                
                # Show first 10 frames
                frames = value.squeeze().cpu().numpy()
                print(f"  First 10 frames: {frames[:10]}")
                
                # Show last 10 frames
                print(f"  Last 10 frames: {frames[-10:]}")
                
                # Check if values are constant
                unique_values = len(np.unique(np.round(frames, 4)))
                print(f"  Unique values (rounded to 4 decimals): {unique_values}/{len(frames)}")
    else:
        print("\n✗ No violin_diagnostics found in outputs!")
    
    print(f"\n{'='*80}")
    print("EXPECTED RANGES (from config)")
    print("="*80)
    helmholtz_config = model_config.get("model", {}).get("helmholtz", {})
    print(f"  β (bow position): [{helmholtz_config.get('β_min', 0.05)}, {helmholtz_config.get('β_max', 0.50)}]")
    print(f"  γ (notch depth): [0, 1]")
    print(f"  α (brightness): [{helmholtz_config.get('α_min', -1.0)}, {helmholtz_config.get('α_max', 1.0)}]")
    print(f"  B (inharmonicity): [0, {helmholtz_config.get('inharmonicity_b_max', 0.0005)}]")

inspect_model_outputs()

Loading model: Bernardel_ear_norm_full__vio
✓ Model loaded

Preprocessed data shapes:
  signals: (4000, 64000)
  pitches: (4000, 1000)
  loudness: (4000, 1000)

Input tensor shapes:
  pitch: torch.Size([1, 1000, 1])
  loudness: torch.Size([1, 1000, 1])
  audio: torch.Size([1, 64000])

Running inference...

OUTPUTS INSPECTION

Top-level keys in outputs:
  violin_diagnostics: dict with 11 keys
  signal: tensor, shape=torch.Size([1, 64000, 1])

VIOLIN_DIAGNOSTICS INSPECTION

Keys in violin_diagnostics:
  - β
  - γ
  - α
  - B
  - residuals
  - bow_notch_smooth
  - bow_mask
  - brightness_tilt
  - spectrum_before_culling
  - nyquist_mask
  - spectrum_after_culling

PARAMETER DETAILS

β:
  Shape: torch.Size([1, 1000, 1])
  Dtype: torch.float32
  Min: 0.052065
  Max: 0.061071
  Mean: 0.054303
  Std: 0.001342
  First 10 frames: [0.06107099 0.05658107 0.05521601 0.05441071 0.05390438 0.05356891
 0.05335864 0.05316894 0.05298413 0.05282797]
  Last 10 frames: [0.05560073 0.0556007  0.05560067 0.